In [1]:
import torch

### set up CUDA as device if available
if torch.cuda.is_available():
    print("GPU is available")
    device = torch.device("cuda")
    cuda_id = torch.cuda.current_device()
    print(f"ID of current CUDA device:{torch.cuda.current_device()}")
    print(f"Name of current CUDA device:{torch.cuda.get_device_name(cuda_id)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("GPU is not available, using CPU")
    device = torch.device("cpu")
print(f"device: {device}")

GPU is available
cuda
CUDA version: 12.4
ID of current CUDA device:0
Name of current CUDA device:NVIDIA H200


In [2]:
import numpy as np
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm, trange
from IPython import display
from sklearn.metrics import average_precision_score
# from conf_and_plot import confusion_matrix_plots
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import KFold
from sklearn.model_selection import ParameterGrid
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import kruskal
import torch.nn.functional as F
from IPython.display import clear_output
import statistics

def seed_all(seed):
    if not seed:
        seed = 10

    print("[ Using Seed : ", seed, " ]")

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_all(2025)

[ Using Seed :  2025  ]


In [3]:
scint_thresh = 0.1 # set the phase scintillation threshold

### following variable not being used anywhere
scint_outlier_thresh = 5. # set the value that determines phase scintillation outliers (these data samples will be removed)

processed_data_2015 = pd.read_csv("processed_data_2015.csv")
processed_data_2015 = processed_data_2015.drop(processed_data_2015.columns[0], axis=1)
predicted_label = 'sigmaPhi projected to vertical at prediction time [radians]'
y = processed_data_2015[predicted_label].values
print(y.shape)

X_fSelect = processed_data_2015.drop(predicted_label, axis=1)
X_fSelect = X_fSelect.values
print(X_fSelect.shape)

(4465846,)
(4465846, 15)


In [4]:
class LSTM_model(nn.Module):
    def __init__(self, hidden_layer_size, num_layers, dropout_p, loss, input_layer_norm, input_size=15, num_classes=1):
        super(LSTM_model, self).__init__()
        self.loss = loss
        self.input_layer_norm = input_layer_norm
        
        self.lstm = nn.LSTM(input_size, hidden_layer_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Linear(hidden_layer_size, num_classes)
        self.sigmoid = nn.Sigmoid()

        self.layer_norm_input = nn.LayerNorm(input_size)
        self.layer_norm_output = nn.LayerNorm(hidden_layer_size)

    def forward(self, x):

        if self.input_layer_norm == True:
            x = self.layer_norm_input(x)
        elif self.input_layer_norm == False:
            pass
        
        lstm_out, _ = self.lstm(x)

        # lstm_out = self.layer_norm_output(lstm_out)
        lstm_out = self.dropout(lstm_out)
        output = self.fc(lstm_out)
        
        if self.loss == 'focal':
            output = self.sigmoid(output)
        elif self.loss == 'bce':
            pass
        return output

In [5]:
def true_skill_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    tss = (TP / (TP + FN)) - (FP / (FP + TN))
    return tss

def heidke_skill_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    numerator = 2 * (TP * TN - FP * FN)
    denominator = (TP + FP) * (FP + TN) + (TP + FN) * (TN + FP)
    
    # Avoid division by zero
    if denominator == 0:
        return 0.0
    
    hss = numerator / denominator
    return hss

In [6]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction  # 'mean', 'sum', or 'none'

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        p_t = inputs * targets + (1 - inputs) * (1 - targets)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        elif self.reduction == 'none':
            return focal_loss
        else:
            raise ValueError(f"Invalid reduction method: {self.reduction}")

In [7]:
def train_model_with_cv(model, train_dataloader, val_dataloader, criterion, optimizer, scheduler, num_epochs):

    epochs = []
    training_loss = []
    validation_loss = []
    training_tss = []
    validation_tss = []
    training_hss = []
    validation_hss = []

    for epoch in range(num_epochs):

        ### training loop
        model.train()
        running_loss = 0.0
        predicted_training_labels = np.array([])
        y_train = np.array([])

        # for batch_inputs, batch_labels in tqdm(train_dataloader):
        for batch_inputs, batch_labels in train_dataloader:
            
            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)
            
            optimizer.zero_grad()
            output = model(batch_inputs)
            
            output.to(device)
            
            loss = criterion(output, batch_labels.float())
            loss.backward()
            optimizer.step()
            
            if scheduler == False:
                pass
            else:
                scheduler.step()
            
            running_loss += loss.item()
            with torch.no_grad(): predicted_training_labels = np.append(predicted_training_labels, output.cpu())
            with torch.no_grad(): y_train = np.append(y_train, batch_labels.cpu())

        predicted_training_labels = np.where(predicted_training_labels > 0.1, 1, 0)

        train_loss = running_loss / len(train_dataloader)
        train_tss = true_skill_score(y_train.astype(int), predicted_training_labels.astype(int))
        train_hss = heidke_skill_score(y_train.astype(int), predicted_training_labels.astype(int))

        ### validation loop
        model.eval()
        with torch.no_grad():
            running_loss = 0.0
            predicted_val_labels = np.array([])
            y_val = np.array([])

            # for batch_inputs, labels in tqdm(val_dataloader):
            for batch_inputs, batch_labels in val_dataloader:
            
                batch_inputs = batch_inputs.to(device)
                batch_labels = batch_labels.to(device)
                
                outputs = model(batch_inputs)
            
                outputs.to(device)
                
                outputs = model(batch_inputs)
                loss = criterion(outputs, batch_labels.float())
                running_loss += loss.item()
                predicted_val_labels = np.append(predicted_val_labels, outputs.cpu())
                y_val = np.append(y_val, batch_labels.cpu())

        predicted_val_labels = np.where(predicted_val_labels > 0.1, 1, 0)

        val_loss = running_loss / len(val_dataloader)
        val_tss = true_skill_score(y_val.astype(int), predicted_val_labels.astype(int))
        val_hss = heidke_skill_score(y_val.astype(int), predicted_val_labels.astype(int))

        epochs.append(epoch)
        training_loss.append(train_loss)
        validation_loss.append(val_loss)
        training_tss.append(train_tss)
        validation_tss.append(val_tss)
        training_hss.append(train_hss)
        validation_hss.append(val_hss)

    return training_hss, validation_hss, training_tss, validation_tss


In [8]:
class LSTMBinaryTimeSeriesClassifier():
    def __init__(self, hidden_layer_size=16, num_layers=2, sequence_length=60, dropout_p=0.1, 
                 batch_size=64, optimizer_type = 'adam', lr=0.001, sgd_momentum=0.9, weight_decay=0.0001, 
                 loss='bce', bce_pos_class_weight=50, focal_alpha=0.25, focal_gamma=2,
                 num_epochs=5, cv_folds=5, scheduler_t=0, input_layer_norm=False):
        self.hidden_layer_size = hidden_layer_size
        self.num_layers = num_layers
        self.sequence_length = sequence_length
        self.dropout_p = dropout_p
        self.batch_size = batch_size
        self.optimizer_type = optimizer_type
        self.lr = lr
        self.weight_decay = weight_decay
        self.sgd_momentum = sgd_momentum
        self.loss = loss
        self.bce_pos_class_weight = bce_pos_class_weight
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.num_epochs = num_epochs
        self.cv_folds = cv_folds
        self.scheduler_t = scheduler_t
        self.input_layer_norm = input_layer_norm
        
    def reshape_data_to_seq_length(self, data, seq_len):
        num_samples = data.shape[0]
        input_features = data.shape[1]
        num_batches = num_samples // seq_len
        data = data[:num_batches * seq_len] # truncate points that don't divide evenly
        reshaped_data = data.reshape(num_batches, seq_len, input_features)
        return reshaped_data

    def reshape_labels_to_seq_length(self, labels, seq_len):
        labels = torch.from_numpy(labels) if isinstance(labels, np.ndarray) else labels
        num_samples = labels.shape[0]
        num_batches = num_samples // seq_len
        labels = labels[:num_batches * seq_len] # truncate points that don't divide evenly
        labels = labels.view(num_batches, seq_len, -1)
        return labels
    
    def fit(self, X, y):

        cv_tss_values = []
        cv_hss_values = []

        tscv = TimeSeriesSplit(n_splits=self.cv_folds)
        for i, (train_index, val_index) in enumerate(tscv.split(X)):
        
            model = LSTM_model(hidden_layer_size=self.hidden_layer_size, 
                               num_layers=self.num_layers, 
                               dropout_p=self.dropout_p, 
                               loss=self.loss, input_layer_norm=self.input_layer_norm)
            model.to(device)
            
            if self.loss == 'bce':
                criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([self.bce_pos_class_weight]).to(device))
            elif self.loss == 'focal':
                criterion = FocalLoss(alpha=self.focal_alpha, gamma=self.focal_gamma)

            if self.optimizer_type == 'adam':
                optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            elif self.optimizer_type == 'sgd':
                optimizer = optim.SGD(model.parameters(), lr=self.lr, momentum=self.sgd_momentum, weight_decay=self.weight_decay)

            if self.scheduler_t > 0:
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.scheduler_t)
            else:
                scheduler = False

            X_train, X_val = X[train_index], X[val_index]
            y_train, y_val = y[train_index], y[val_index]
            
            scaler_X = RobustScaler()
            scaler_X = scaler_X.fit(X_train)

            X_train_scaled = scaler_X.transform(X_train)
            X_val_scaled = scaler_X.transform(X_val)
            
            X_train_scaled = self.reshape_data_to_seq_length(X_train_scaled, self.sequence_length)
            X_val_scaled = self.reshape_data_to_seq_length(X_val_scaled, self.sequence_length)

            train_labels = self.reshape_labels_to_seq_length(y_train, self.sequence_length)
            val_labels = self.reshape_labels_to_seq_length(y_val, self.sequence_length)
            
            train_inputs = torch.tensor(X_train_scaled, dtype=torch.float32)#.transpose(1, 2)  # Shape: [batch_size, 15, 1]
            train_inputs = train_inputs.to(device)
            train_labels = train_labels.to(device)
            train_dataset = TensorDataset(train_inputs, train_labels)
            train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=False, worker_init_fn=seed_worker)

            val_inputs = torch.tensor(X_val_scaled, dtype=torch.float32)#.transpose(1, 2)  # Shape: [batch_size, 15, 1]
            val_inputs = val_inputs.to(device)
            val_labels = val_labels.to(device)
            val_dataset = TensorDataset(val_inputs, val_labels)
            val_dataloader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False, worker_init_fn=seed_worker)
            
            training_hss, validation_hss, training_tss, validation_tss = \
            train_model_with_cv(model, 
                 train_dataloader, val_dataloader, 
                 criterion, optimizer, scheduler,
                 num_epochs=self.num_epochs)
            cv_tss_values.append(validation_tss[-1])
            cv_hss_values.append(validation_hss[-1])

            if i == 0:
                total_params = sum(p.numel() for p in model.parameters())
                print(f'Total number of parameters: {total_params}')
                
            print(f"Fold {i}: train tss = {training_tss[-1]:.4f} : val tss = {validation_tss[-1]:.4f} ::: train hss = {training_hss[-1]:.4f} : val hss = {validation_hss[-1]:.4f}")
        
        return cv_hss_values, cv_tss_values
    

In [9]:
training_data_size = 50000

X_train, X_test, \
    y_train, y_test, \
        idx_train, idx_test = train_test_split(X_fSelect, y, range(len(y)), train_size=training_data_size, shuffle=False)

### strategic hypertune iteration 1
param_grid = {
    'hidden_layer_size': [4, 16],
    'num_layers': [2, 4],
    'sequence_length': [15, 45],
    'dropout_p': [0.1, 0.5],
    'batch_size': [32, 512],
    'lr': [0.0001, 0.01],
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.0001, 0.01],
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 75],
    'input_layer_norm':[True, False],
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 2
param_grid = {
    'hidden_layer_size': [2, 8],  ### changed
    'num_layers': [1, 4],  ### changed
    'sequence_length': [15, 45],
    'dropout_p': [0.1, 0.5],
    'batch_size': [256, 1024],  ### changed
    'lr': [0.005, 0.01],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.0001, 0.01],
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 75],
    'input_layer_norm':[True, False],
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 3
param_grid = {
    'hidden_layer_size': [2, 4, 12],  ### changed, added 1 dimension
    'num_layers': [1, 2],  ### changed
    'sequence_length': [30, 45],  ### changed
    'dropout_p': [0.1, 0.5],
    'batch_size': [128, 256],  ### changed
    'lr': [0.005, 0.01],  
    'optimizer_type': ['adam'],  ### changed, removed 1 dimension
    'weight_decay':[0.0001, 0.01],
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 75],
    'input_layer_norm':[True, False],
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 4
param_grid = {
    'hidden_layer_size': [2, 4, 12],  
    'num_layers': [1, 2],  
    'sequence_length': [40, 45],  ### changed
    'dropout_p': [0.1, 0.5],
    'batch_size': [128, 256],  
    'lr': [0.005, 0.01],  
    'optimizer_type': ['adam'],  
    'weight_decay':[0.001, 0.01],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 50],  ### changed
    'input_layer_norm':[True, False],
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 5
param_grid = {
    'hidden_layer_size': [2, 4, 16],  ### changed
    'num_layers': [1, 2],  
    'sequence_length': [15, 30, 45],  ### changed, added 1 dimension
    'dropout_p': [0.1, 0.5],
    'batch_size': [128, 256],  
    'lr': [0.005, 0.0075],  ### changed
    'optimizer_type': ['adam'],  
    'weight_decay':[0.0075, 0.025],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [15, 30],  ### changed
    'input_layer_norm':[False],  ### changed, removed 1 dimension
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 6
param_grid = {
    'hidden_layer_size': [2, 4, 32],  ### changed
    'num_layers': [1, 2],  
    'sequence_length': [15, 40, 45],  ### changed
    'dropout_p': [0.1, 0.5],
    'batch_size': [128, 256],  
    'lr': [0.005, 0.0075],  
    'optimizer_type': ['adam'],  
    'weight_decay':[0.01, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [15, 30, 45],  ### changed, added 1 dimension
    'input_layer_norm':[False],  
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 7
param_grid = {
    'hidden_layer_size': [2, 4, 8],  ### changed
    'num_layers': [1, 2],  
    'sequence_length': [15, 40, 45],  
    'dropout_p': [0.1, 0.3],  ### changed
    'batch_size': [128, 256],  
    'lr': [0.0025, 0.01],  ### changed
    'optimizer_type': ['adam'],  
    'weight_decay':[0.005, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [20, 40, 60],  ### changed
    'input_layer_norm':[False],  
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 8
param_grid = {
    'hidden_layer_size': [2, 4, 64],  ### changed
    'num_layers': [1, 2],  
    'sequence_length': [15, 40, 45],  
    'dropout_p': [0.1, 0.3],  
    'batch_size': [128, 256],  
    'lr': [0.01, 0.025],  ### changed
    'optimizer_type': ['adam'],  
    'weight_decay':[0.001, 0.075],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [15, 35, 65],  ### changed
    'input_layer_norm':[False],  
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 9
param_grid = {
    'hidden_layer_size': [2, 4, 24],  ### changed
    'num_layers': [1, 2],  
    'sequence_length': [25, 35, 45],  ### changed
    'dropout_p': [0.1, 0.2],  ### changed
    'batch_size': [128, 256],  
    'lr': [0.005, 0.01],  ### changed
    'optimizer_type': ['adam'],  
    'weight_decay':[0.01, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [10, 30, 60],  ### changed
    'input_layer_norm':[False],  
    'num_epochs': [50],
    'cv_folds': [5], 
}

### strategic hypertune iteration 10
param_grid = {
    'hidden_layer_size': [2, 4, 16],  ### changed
    'num_layers': [1, 2],  
    'sequence_length': [10, 35, 45],  ### changed
    'dropout_p': [0.1, 0.2],  
    'batch_size': [128, 256],  
    'lr': [0.0065, 0.015],  ### changed
    'optimizer_type': ['adam'],  
    'weight_decay':[0.01, 0.05],  
    'loss': ['bce'],
    'bce_pos_class_weight': [5, 50, 80],  ### changed
    'input_layer_norm':[False],  
    'num_epochs': [50],
    'cv_folds': [5], 
}

param_combinations = list(ParameterGrid(param_grid))

grid = pd.DataFrame()
tss_distributions = []
hss_distributions = []
mean_tss_scores = []
mean_hss_scores = []
for params in param_combinations:

    print(str(param_combinations.index(params)) + "/" + str(len(param_combinations)))
    print(params)

    lstm_time_series_classifier = LSTMBinaryTimeSeriesClassifier(**params)
    cv_hss_values, cv_tss_values = lstm_time_series_classifier.fit(X_train, y_train)
    
    tss_distributions.append(cv_tss_values)
    hss_distributions.append(cv_hss_values)
    mean_cv_tss = sum(cv_tss_values)/len(cv_tss_values)
    mean_cv_hss = sum(cv_hss_values)/len(cv_hss_values)
    print(f"mean cv TSS: {mean_cv_tss:.4f}")
    print(f"mean cv HSS: {mean_cv_hss:.4f}")
    mean_tss_scores.append(mean_cv_tss)
    mean_hss_scores.append(mean_cv_hss)
    
    if params == param_combinations[0]:
        grid = pd.DataFrame(params, index=[0])
    else:
        grid = pd.concat([grid, pd.DataFrame([params])], ignore_index=True)

grid['tss'] = [round(num, 4) for num in mean_tss_scores]
grid['hss'] = [round(num, 4) for num in mean_hss_scores]
grid = grid.sort_values(by=['tss', 'hss'], ascending=False)
grid = grid.drop(columns=[col for col in ['num_epochs','cv_folds','loss','scheduler_t'] if col in grid.columns], axis=1)
grid.head(20)


0/864
{'batch_size': 128, 'bce_pos_class_weight': 5, 'cv_folds': 5, 'dropout_p': 0.1, 'hidden_layer_size': 2, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.0065, 'num_epochs': 50, 'num_layers': 1, 'optimizer_type': 'adam', 'sequence_length': 10, 'weight_decay': 0.01}
Total number of parameters: 189
Fold 0: train tss = 0.0500 : val tss = 0.0181 ::: train hss = 0.0952 : val hss = 0.0330
Fold 1: train tss = 0.5799 : val tss = 0.3466 ::: train hss = 0.6206 : val hss = 0.2443
Fold 2: train tss = 0.3356 : val tss = 0.6738 ::: train hss = 0.3479 : val hss = 0.5135
Fold 3: train tss = 0.5618 : val tss = 0.0262 ::: train hss = 0.4781 : val hss = 0.0297
Fold 4: train tss = 0.4661 : val tss = 0.4340 ::: train hss = 0.4137 : val hss = 0.4481
mean cv TSS: 0.2997
mean cv HSS: 0.2537
1/864
{'batch_size': 128, 'bce_pos_class_weight': 5, 'cv_folds': 5, 'dropout_p': 0.1, 'hidden_layer_size': 2, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.0065, 'num_epochs': 50, 'num_layers': 1, 'optimizer_type

Fold 1: train tss = 0.5374 : val tss = 0.1325 ::: train hss = 0.6654 : val hss = 0.0896
Fold 2: train tss = 0.5712 : val tss = 0.6614 ::: train hss = 0.4550 : val hss = 0.4261
Fold 3: train tss = 0.5916 : val tss = 0.0720 ::: train hss = 0.4687 : val hss = 0.0819
Fold 4: train tss = 0.6607 : val tss = 0.4666 ::: train hss = 0.4100 : val hss = 0.3914
mean cv TSS: 0.2665
mean cv HSS: 0.1978
11/864
{'batch_size': 128, 'bce_pos_class_weight': 5, 'cv_folds': 5, 'dropout_p': 0.1, 'hidden_layer_size': 2, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.0065, 'num_epochs': 50, 'num_layers': 2, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.05}
Total number of parameters: 237
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 2: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 3: train tss = 0.5205 : val tss = 

Total number of parameters: 237
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.2691 : val tss = 0.0000 ::: train hss = 0.3717 : val hss = 0.0000
Fold 2: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 3: train tss = 0.2245 : val tss = -0.0004 ::: train hss = 0.2942 : val hss = -0.0007
Fold 4: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
mean cv TSS: -0.0001
mean cv HSS: -0.0001
22/864
{'batch_size': 128, 'bce_pos_class_weight': 5, 'cv_folds': 5, 'dropout_p': 0.1, 'hidden_layer_size': 2, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.015, 'num_epochs': 50, 'num_layers': 2, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.01}
Total number of parameters: 237
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.6230 : val tss = 0.2938 ::: train hss = 0.4529 : val hss = 0.2264
Fold

Fold 4: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
mean cv TSS: -0.0003
mean cv HSS: -0.0005
32/864
{'batch_size': 128, 'bce_pos_class_weight': 5, 'cv_folds': 5, 'dropout_p': 0.1, 'hidden_layer_size': 4, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.0065, 'num_epochs': 50, 'num_layers': 2, 'optimizer_type': 'adam', 'sequence_length': 35, 'weight_decay': 0.01}
Total number of parameters: 539
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.6876 : val tss = 0.3394 ::: train hss = 0.6724 : val hss = 0.2383
Fold 2: train tss = 0.5360 : val tss = 0.6317 ::: train hss = 0.4267 : val hss = 0.4442
Fold 3: train tss = 0.7196 : val tss = 0.0325 ::: train hss = 0.5025 : val hss = 0.0501
Fold 4: train tss = 0.5620 : val tss = 0.4906 ::: train hss = 0.4701 : val hss = 0.3818
mean cv TSS: 0.2988
mean cv HSS: 0.2229
33/864
{'batch_size': 128, 'bce_pos_class_weight': 5, 'cv_folds': 5, 'dropout_p': 0.1

Total number of parameters: 539
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.5480 : val tss = 0.2481 ::: train hss = 0.4994 : val hss = 0.1959
Fold 2: train tss = 0.0958 : val tss = 0.4672 ::: train hss = 0.1174 : val hss = 0.3677
Fold 3: train tss = 0.5639 : val tss = 0.0370 ::: train hss = 0.4282 : val hss = 0.0320
Fold 4: train tss = 0.3908 : val tss = 0.5485 ::: train hss = 0.3052 : val hss = 0.3878
mean cv TSS: 0.2602
mean cv HSS: 0.1967
43/864
{'batch_size': 128, 'bce_pos_class_weight': 5, 'cv_folds': 5, 'dropout_p': 0.1, 'hidden_layer_size': 4, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.015, 'num_epochs': 50, 'num_layers': 2, 'optimizer_type': 'adam', 'sequence_length': 10, 'weight_decay': 0.05}
Total number of parameters: 539
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 2: 

...

Fold 4: train tss = 0.6982 : val tss = 0.6175 ::: train hss = 0.1493 : val hss = 0.1905
mean cv TSS: 0.4342
mean cv HSS: 0.1408
822/864
{'batch_size': 256, 'bce_pos_class_weight': 80, 'cv_folds': 5, 'dropout_p': 0.2, 'hidden_layer_size': 4, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.0065, 'num_epochs': 50, 'num_layers': 2, 'optimizer_type': 'adam', 'sequence_length': 10, 'weight_decay': 0.01}
Total number of parameters: 539
Fold 0: train tss = 0.6734 : val tss = 0.1915 ::: train hss = 0.1056 : val hss = 0.0833
Fold 1: train tss = 0.8708 : val tss = 0.3058 ::: train hss = 0.3394 : val hss = 0.1368
Fold 2: train tss = 0.7561 : val tss = 0.6248 ::: train hss = 0.1709 : val hss = 0.1938
Fold 3: train tss = 0.7906 : val tss = 0.1040 ::: train hss = 0.2978 : val hss = 0.0219
Fold 4: train tss = 0.7399 : val tss = 0.6281 ::: train hss = 0.1796 : val hss = 0.2158
mean cv TSS: 0.3708
mean cv HSS: 0.1303
823/864
{'batch_size': 256, 'bce_pos_class_weight': 80, 'cv_folds': 5, 'dropout_p': 0

Fold 1: train tss = 0.7806 : val tss = 0.4603 ::: train hss = 0.1653 : val hss = 0.2018
Fold 2: train tss = 0.7502 : val tss = 0.6234 ::: train hss = 0.1614 : val hss = 0.1971
Fold 3: train tss = 0.6983 : val tss = 0.0878 ::: train hss = 0.1525 : val hss = 0.0097
Fold 4: train tss = 0.6914 : val tss = 0.5941 ::: train hss = 0.1400 : val hss = 0.1946
mean cv TSS: 0.4174
mean cv HSS: 0.1402
833/864
{'batch_size': 256, 'bce_pos_class_weight': 80, 'cv_folds': 5, 'dropout_p': 0.2, 'hidden_layer_size': 4, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.015, 'num_epochs': 50, 'num_layers': 1, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.05}
Total number of parameters: 379
Fold 0: train tss = 0.5221 : val tss = 0.3115 ::: train hss = 0.0794 : val hss = 0.1390
Fold 1: train tss = 0.8043 : val tss = 0.3504 ::: train hss = 0.1777 : val hss = 0.0844
Fold 2: train tss = 0.6914 : val tss = 0.5728 ::: train hss = 0.1329 : val hss = 0.1664
Fold 3: train tss = 0.7294 : val tss =

Total number of parameters: 2191
Fold 0: train tss = 0.6629 : val tss = 0.3259 ::: train hss = 0.0775 : val hss = 0.1268
Fold 1: train tss = 0.7918 : val tss = 0.3068 ::: train hss = 0.1438 : val hss = 0.0571
Fold 2: train tss = 0.7511 : val tss = 0.6830 ::: train hss = 0.1883 : val hss = 0.2550
Fold 3: train tss = 0.7398 : val tss = 0.2912 ::: train hss = 0.1973 : val hss = 0.0365
Fold 4: train tss = 0.7193 : val tss = 0.6421 ::: train hss = 0.1676 : val hss = 0.2434
mean cv TSS: 0.4498
mean cv HSS: 0.1438
844/864
{'batch_size': 256, 'bce_pos_class_weight': 80, 'cv_folds': 5, 'dropout_p': 0.2, 'hidden_layer_size': 16, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.0065, 'num_epochs': 50, 'num_layers': 1, 'optimizer_type': 'adam', 'sequence_length': 45, 'weight_decay': 0.01}
Total number of parameters: 2191
Fold 0: train tss = 0.7166 : val tss = 0.2578 ::: train hss = 0.0919 : val hss = 0.1030
Fold 1: train tss = 0.7650 : val tss = 0.3758 ::: train hss = 0.2125 : val hss = 0.1455
Fo

Fold 4: train tss = 0.6782 : val tss = 0.6310 ::: train hss = 0.1428 : val hss = 0.1913
mean cv TSS: 0.4456
mean cv HSS: 0.1265
854/864
{'batch_size': 256, 'bce_pos_class_weight': 80, 'cv_folds': 5, 'dropout_p': 0.2, 'hidden_layer_size': 16, 'input_layer_norm': False, 'loss': 'bce', 'lr': 0.015, 'num_epochs': 50, 'num_layers': 1, 'optimizer_type': 'adam', 'sequence_length': 35, 'weight_decay': 0.01}
Total number of parameters: 2191
Fold 0: train tss = 0.9202 : val tss = 0.0209 ::: train hss = 0.1282 : val hss = 0.0121
Fold 1: train tss = 0.8516 : val tss = 0.3425 ::: train hss = 0.2598 : val hss = 0.1287
Fold 2: train tss = 0.7562 : val tss = 0.6460 ::: train hss = 0.1296 : val hss = 0.2355
Fold 3: train tss = 0.8066 : val tss = 0.0989 ::: train hss = 0.2662 : val hss = 0.0166
Fold 4: train tss = 0.7857 : val tss = 0.6657 ::: train hss = 0.2078 : val hss = 0.2471
mean cv TSS: 0.3548
mean cv HSS: 0.1280
855/864
{'batch_size': 256, 'bce_pos_class_weight': 80, 'cv_folds': 5, 'dropout_p': 

,batch_size,bce_pos_class_weight,dropout_p,hidden_layer_size,input_layer_norm,lr,num_layers,optimizer_type,sequence_length,weight_decay,tss,hss
723,256,80,0.1,2,False,0.0065,1,adam,35,0.05,0.5235,0.1588
291,128,80,0.1,2,False,0.0065,1,adam,35,0.05,0.5161,0.1694
663,256,50,0.2,2,False,0.0150,1,adam,35,0.05,0.5098,0.2064
662,256,50,0.2,2,False,0.0150,1,adam,35,0.01,0.5071,0.2105
615,256,50,0.1,4,False,0.0150,1,adam,35,0.05,0.5069,0.2061
399,128,80,0.2,4,False,0.0150,1,adam,35,0.05,0.5041,0.1606
316,128,80,0.1,4,False,0.0065,1,adam,45,0.01,0.4928,0.1519
183,128,50,0.1,4,False,0.0150,1,adam,35,0.05,0.4922,0.1843
150,128,50,0.1,2,False,0.0065,2,adam,10,0.01,0.4910,0.1790
216,128,50,0.2,2,False,0.0065,1,adam,10,0.01,0.4906,0.1885


In [10]:
grid.sort_values(by=['hss', 'tss'], ascending=False).head(20)

,batch_size,bce_pos_class_weight,dropout_p,hidden_layer_size,input_layer_norm,lr,num_layers,optimizer_type,sequence_length,weight_decay,tss,hss
40,128,5,0.1,4,False,0.0150,1,adam,45,0.01,0.2959,0.2975
75,128,5,0.2,2,False,0.0065,1,adam,35,0.05,0.3457,0.2896
76,128,5,0.2,2,False,0.0065,1,adam,45,0.01,0.3626,0.2878
50,128,5,0.1,16,False,0.0065,1,adam,35,0.01,0.3350,0.2823
130,128,5,0.2,16,False,0.0065,2,adam,45,0.01,0.3206,0.2727
570,256,5,0.2,16,False,0.0150,2,adam,10,0.01,0.3160,0.2717
593,256,50,0.1,2,False,0.0150,1,adam,45,0.05,0.4789,0.2713
496,256,5,0.1,16,False,0.0150,1,adam,45,0.01,0.3310,0.2679
566,256,5,0.2,16,False,0.0150,1,adam,35,0.01,0.3188,0.2679
110,128,5,0.2,4,False,0.0150,1,adam,35,0.01,0.3017,0.2675


In [11]:
if grid.empty:
    print("grid empty")
else:
    grid.to_csv("1_lstm_hypertune_grid_output_10.csv")
    print("grid output successful, saved as 1_lstm_hypertune_grid_output_10.csv")
    

grid output successful, saved as 1_lstm_hypertune_grid_output_10.csv
